# Multi-source ingestion

Four live open sources, no API keys. Replaces `scripts/ingest.py`.

**This notebook ingests a deliberately small slice** so it executes in under a minute. The graph it reads already holds ~6,850 nodes from seven therapeutic areas; use the pipeline functions directly for a bulk load.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

# Fail fast with an actionable message rather than a driver error.
config.require('neo4j')

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## Fetch from the live APIs

ClinicalTrials.gov v2 · openFDA FAERS · HGNC. The free-text intervention and `medicinalproduct` fields are the point — that is messiness we did not author.

### The loaders

Inlined rather than imported, so the Cypher is visible where it runs. `MERGE` on natural keys, so re-running converges.

In [2]:
# Graph loading. Defined here rather than imported, so the Cypher is visible
# where it runs. MERGE on natural keys, so re-running converges.

GENES = ["BRAF", "NRAS", "KIT", "PTEN", "TP53", "CDKN2A", "MITF", "TERT"]

SCHEMA = [
    "CREATE CONSTRAINT re_id IF NOT EXISTS FOR (e:ResolvedEntity) REQUIRE e.entity_id IS UNIQUE",
    "CREATE CONSTRAINT men_nat IF NOT EXISTS FOR (m:Mention) "
    "REQUIRE (m.source, m.source_record_id, m.raw_text) IS UNIQUE",
    "CREATE CONSTRAINT ident_nat IF NOT EXISTS FOR (i:Identifier) "
    "REQUIRE (i.authority, i.value, i.release) IS UNIQUE",
    "CREATE CONSTRAINT trial_nct IF NOT EXISTS FOR (t:ClinicalTrial) REQUIRE t.nct_id IS UNIQUE",
    "CREATE CONSTRAINT gene_hgnc IF NOT EXISTS FOR (g:Gene) REQUIRE g.hgnc_id IS UNIQUE",
    "CREATE CONSTRAINT aer_id IF NOT EXISTS FOR (a:AdverseEventReport) REQUIRE a.report_id IS UNIQUE",
    "CREATE CONSTRAINT rxn_term IF NOT EXISTS FOR (r:Reaction) REQUIRE r.term IS UNIQUE",
    "CREATE INDEX men_text IF NOT EXISTS FOR (m:Mention) ON (m.raw_text)",
]


def ensure_schema(c):
    for stmt in SCHEMA:
        c.run(stmt)
    return c.constraint_count()


def load_trials(c, trials):
    """Trials and their conditions. Intervention strings become Mentions —
    raw, unmodified, awaiting resolution."""
    rows = [{"nct_id": t.nct_id, "title": t.title, "phase": t.phase, "status": t.status,
             "enrollment": t.enrollment, "conditions": list(t.conditions),
             "interventions": list(t.interventions), "sv": t.source_version,
             "at": t.retrieved_at.isoformat()} for t in trials]
    c.run("""
        UNWIND $rows AS r
        MERGE (t:ResolvedEntity:ClinicalTrial:Provenanced {entity_id: 'trial:' + r.nct_id})
          SET t.nct_id = r.nct_id, t.title = r.title, t.phase = r.phase,
              t.status = r.status, t.enrollment = r.enrollment,
              t.source = 'clinicaltrials.gov', t.source_version = r.sv,
              t.retrieved_at = r.at, t.tier = 'ingested'
        WITH t, r
        UNWIND r.conditions AS cond
          MERGE (d:ResolvedEntity:Disease:Provenanced {entity_id: 'disease:' + toLower(cond)})
            SET d.name = cond, d.source = 'clinicaltrials.gov', d.source_version = r.sv,
                d.retrieved_at = r.at, d.tier = 'ingested'
          MERGE (t)-[:STUDIES]->(d)
    """, rows=rows)
    c.run("""
        UNWIND $rows AS r
        MATCH (t:ClinicalTrial {nct_id: r.nct_id})
        UNWIND r.interventions AS iv
          MERGE (m:Mention:Provenanced {source: 'clinicaltrials.gov',
                                        source_record_id: r.nct_id, raw_text: iv})
            SET m.mention_type = 'intervention', m.source_version = r.sv,
                m.retrieved_at = r.at, m.tier = 'ingested'
          MERGE (t)-[:MENTIONS]->(m)
    """, rows=rows)
    return len(rows)


def load_adverse_events(c, aes):
    """FAERS reports. `medicinalproduct` is human-entered — brands, misspellings,
    trailing dosages — and becomes a Mention, never a resolved drug."""
    rows = [{"report_id": a.report_id, "serious": a.serious,
             "drug_names": list(a.drug_names), "reactions": list(a.reactions),
             "sv": a.source_version, "at": a.retrieved_at.isoformat()} for a in aes]
    c.run("""
        UNWIND $rows AS r
        MERGE (a:AdverseEventReport:Provenanced {report_id: r.report_id})
          SET a.serious = r.serious, a.source = 'openfda.faers',
              a.source_version = r.sv, a.retrieved_at = r.at, a.tier = 'ingested'
        WITH a, r
        UNWIND r.reactions AS rx
          MERGE (x:Reaction:Provenanced {term: rx})
            SET x.source = 'openfda.faers', x.source_version = r.sv,
                x.retrieved_at = r.at, x.tier = 'ingested'
          MERGE (a)-[:REPORTS_REACTION]->(x)
    """, rows=rows)
    c.run("""
        UNWIND $rows AS r
        MATCH (a:AdverseEventReport {report_id: r.report_id})
        UNWIND r.drug_names AS dn
          MERGE (m:Mention:Provenanced {source: 'openfda.faers',
                                        source_record_id: r.report_id, raw_text: dn})
            SET m.mention_type = 'medicinalproduct', m.source_version = r.sv,
                m.retrieved_at = r.at, m.tier = 'ingested'
          MERGE (a)-[:MENTIONS]->(m)
    """, rows=rows)
    return len(rows)


def load_genes(c, genes):
    """Genes, plus their PREVIOUS symbols as REPLACED_BY edges — the raw material
    for the schema-evolution demo in notebook 07."""
    rows = [{"hgnc_id": g.hgnc_id, "symbol": g.symbol, "name": g.name,
             "prev": list(g.prev_symbols), "sv": g.source_version,
             "at": g.retrieved_at.isoformat()} for g in genes]
    c.run("""
        UNWIND $rows AS r
        MERGE (g:ResolvedEntity:Gene:Provenanced {entity_id: 'gene:' + r.symbol})
          SET g.hgnc_id = r.hgnc_id, g.symbol = r.symbol, g.name = r.name,
              g.source = 'hgnc', g.source_version = r.sv,
              g.retrieved_at = r.at, g.tier = 'ingested'
        MERGE (i:Identifier:Provenanced {authority:'hgnc', value:r.hgnc_id, release:r.sv})
          SET i.source = 'hgnc', i.source_version = r.sv, i.retrieved_at = r.at,
              i.tier = 'ingested'
        MERGE (g)-[:HAS_IDENTIFIER]->(i)
        WITH g, r
        UNWIND r.prev AS ps
          MERGE (old:Identifier:Provenanced {authority:'hgnc-symbol', value: ps,
                                             release:'historical'})
            SET old.source = 'hgnc', old.source_version = r.sv, old.retrieved_at = r.at,
                old.tier = 'ingested', old.withdrawn = true
          MERGE (old)-[:REPLACED_BY {change_kind:'renamed'}]->(g)
    """, rows=rows)
    return len(rows)

print("loaders defined:", [f.__name__ for f in (ensure_schema, load_trials,
                                                load_adverse_events, load_genes)])

loaders defined: ['ensure_schema', 'load_trials', 'load_adverse_events', 'load_genes']


In [3]:
from biomedical.sources import adapters

print("RxNorm release:", adapters.rxnorm_version())

trials = adapters.fetch_trials("melanoma", limit=10)
aes = adapters.fetch_adverse_events('patient.reaction.reactionmeddrapt:"melanoma"', limit=10)
genes = adapters.fetch_genes(GENES)

print(f"\nclinicaltrials.gov  {len(trials)} trials")
print(f"openfda faers       {len(aes)} reports")
print(f"hgnc                {len(genes)} genes")

print("\nsample trial:")
t = trials[0]
print(f"  {t.nct_id}  {t.phase}  {t.status}")
print(f"  conditions:    {list(t.conditions)[:3]}")
print(f"  interventions: {list(t.interventions)[:3]}   <- free text, no identifier")

print("\nsample FAERS drug names (human-entered):")
for a in aes[:4]:
    print("  ", list(a.drug_names)[:3])

RxNorm release: 05-Oct-2026



clinicaltrials.gov  10 trials
openfda faers       10 reports
hgnc                8 genes

sample trial:
  NCT04106167  NA  TERMINATED
  conditions:    ['Advanced Solid Tumor', 'Lymphoma', 'Gastric Cancer']
  interventions: ['Allogeneic natural killer (NK) cell']   <- free text, no identifier

sample FAERS drug names (human-entered):
   ['REVLIMID']
   ['USTEKINUMAB']
   ['CICLOSPORIN', 'PREDNISONE', 'CELLCEPT']
   ['ADALIMUMAB', 'CYCLOSPORINE', 'REMICADE']


## Load into Neo4j

Idempotent on natural keys, so re-running converges.

In [4]:
from biomedical.graph.store import GraphClient
cfg = config.load()

with GraphClient(uri=cfg.neo4j.uri, username=cfg.neo4j.username,
                 password=cfg.neo4j.password.reveal(), database=cfg.neo4j.database) as c:
    print("constraints:", ensure_schema(c))
    print("trials: ", load_trials(c, trials))
    print("reports:", load_adverse_events(c, aes))
    print("genes:  ", load_genes(c, genes))

constraints: 11


trials:  10


reports: 10


genes:   8


## Graph composition

Counts are reported **per tier and per source**, never as one total, so a reader can see how much is real.

In [5]:
from biomedical.ui import data

o = data.overview()
print(f"nodes {o['nodes']:,}   relationships {o['rels']:,}\n")
print("by label:")
for r in o["labels"]:
    print(f"  {r['label']:<22} {r['count']:>6,}")
print("\nby source:")
for r in o["sources"]:
    print(f"  {r['source']:<22} {str(r['version'])[:14]:<14} {r['count']:>6,}")
print("\nby provenance tier:")
for r in o["tiers"]:
    print(f"  {r['tier']:<22} {r['count']:>6,}")

nodes 7,819   relationships 10,630

by label:
  Mention                 3,714
  ClinicalTrial           1,208
  Disease                 1,133
  Reaction                  526
  Identifier                451
  AdverseEventReport        409
  Drug                      333
  Conflict                   37
  Gene                        8

by source:
  clinicaltrials.gov     v2              4,233
  openfda.faers          faers           2,146
  mondo                  2026-10-06        908
  rxnorm                 rxnav             472
  conflict-detector      1.0                37
  hgnc                   hgnc-current       23

by provenance tier:
  derived                    37
  ingested                7,782


## The real/synthetic boundary

Stated, not implied. Licence-restricted vocabularies (MedDRA, SNOMED CT, UMLS, DrugBank) are excluded by design so a reader never needs a licence to reproduce this.

In [6]:
print("REAL      drug, trial, adverse-event, gene, disease facts")
print("SYNTHETIC patients, consent status, internal governance policies")
print()
print("relationships by type:")
for r in o["rels_by_type"]:
    print(f"  {r['type']:<26} {r['count']:>6,}")

REAL      drug, trial, adverse-event, gene, disease facts
SYNTHETIC patients, consent status, internal governance policies

relationships by type:
  MENTIONS                    4,915
  STUDIES                     2,267
  REPORTS_REACTION            1,245
  CANDIDATE_FOR                 727
  RESOLVES_TO                   556
  HAS_IDENTIFIER                444
  ASSOCIATED_WITH_REACTION      398
  INVESTIGATES                   71
  REPLACED_BY                     7
